In [21]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [38]:
import os
import json
import numpy as np
import pandas as pd
from typing import Dict, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F

In [23]:
base_dir = "/content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M"
preprocessed_dir = os.path.join(base_dir, "AntM2C_preprocessed")

# ID mappings (from 02_id_encoder.ipynb)
id_mappings_dir = os.path.join(preprocessed_dir, "id_mappings")
user_id_2idx_path = os.path.join(id_mappings_dir, "user_id_2idx.json")
item_id_2idx_path = os.path.join(id_mappings_dir, "item_id_2idx.json")
scene_2idx_path = os.path.join(id_mappings_dir, "scene_2idx.json")
id_vocab_config_path = os.path.join(id_mappings_dir, "id_vocab_config.json")

# Precomputed embeddings
text_embeddings_path = os.path.join(preprocessed_dir, "text_embeddings.npy")               # (N_items, 768)
image_embeddings_aligned_path = os.path.join(base_dir, "image_embeddings_vit_base_aligned.pt")
image_missing_mask_path = os.path.join(base_dir, "image_missing_mask.pt")
time_features_path = os.path.join(preprocessed_dir, "time_features_final.pt")              # (N_interactions, D_time)
time_feature_cols_path = os.path.join(preprocessed_dir, "time_features_columns.json")      # list of column names

# fusion artifacts from this notebook will be saved here
fusion_artifacts_dir = os.path.join(base_dir, "fusion_artifacts")
os.makedirs(fusion_artifacts_dir, exist_ok=True)

In [24]:
# sanity check
print("base_dir:", base_dir)
print("preprocessed_dir:", preprocessed_dir)
print("id_mappings_dir:", id_mappings_dir)
print("text_embeddings_path:", text_embeddings_path)
print("image_embeddings_path:", image_embeddings_aligned_path)
print("time_features_path:", time_features_path)
print("fusion_artifacts_dir:", fusion_artifacts_dir)

base_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M
preprocessed_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed
id_mappings_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed/id_mappings
text_embeddings_path: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed/text_embeddings.npy
image_embeddings_path: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/image_embeddings_vit_base_aligned.pt
time_features_path: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed/time_features_final.pt
fusion_artifacts_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/fusion_artifacts


In [25]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [26]:
# Next, we will load Embedding Sizes (Dynamic Sizing)
# Load shapes without pulling everything into GPU memory
text_embeddings = np.load(text_embeddings_path, mmap_mode="r")
text_dim = text_embeddings.shape[1]

image_embeddings = torch.load(image_embeddings_aligned_path, map_location="cpu")
image_dim = image_embeddings.shape[1]

time_features = torch.load(time_features_path, map_location="cpu")
time_raw_dim = time_features.shape[1]

In [27]:
# Load time feature names
with open(time_feature_cols_path, "r") as f:
    time_feature_cols = json.load(f)

print("--- Embedding/Feature Dimensions ---")
print("Text embedding dim:", text_dim)
print("Image embedding dim:", image_dim)
print("Time raw feature dim:", time_raw_dim)
print("Time feature columns:", time_feature_cols)

--- Embedding/Feature Dimensions ---
Text embedding dim: 1536
Image embedding dim: 768
Time raw feature dim: 5
Time feature columns: ['unix_time_norm', 'hour_sin', 'hour_cos', 'dayofweek', 'is_weekend']


In [28]:
# Load vocab sizes (for ID embeddings)
with open(id_vocab_config_path, "r") as f:
    id_vocab_config = json.load(f)

user_vocab_size = int(id_vocab_config["num_users"])
item_vocab_size = int(id_vocab_config["num_items"])
scene_vocab_size = int(id_vocab_config["num_scenes"])

print("\n--- ID Vocabulary Sizes ---")
print("user_vocab_size:", user_vocab_size)
print("item_vocab_size:", item_vocab_size)
print("scene_vocab_size:", scene_vocab_size)


--- ID Vocabulary Sizes ---
user_vocab_size: 67615
item_vocab_size: 59142
scene_vocab_size: 5


## Load All Precomputed Embeddings / Feature Shapes

In this step, we validate and summarize the shapes of all modality inputs that will enter the ColdFusionX fusion module.
These inputs reference back to earlier notebooks:
*   Text embeddings from 03_text_encoder_BERT.ipynb
*   Image embeddings from 04_image_encoder.ipynb (ViT-base CLS, 768-d)
*   Time features from 05_time_encoder.ipynb (raw time vector, 5-d)
*   ID vocab sizes from 02_id_encoder.ipynb (user/item/scene mappings)

To keep memory usage low, we first load arrays/tensors in a lightweight way (e.g., NumPy memmap, CPU tensors).
We then run sanity checks to ensure dimensions match expected vocab sizes and that indexing will remain consistent during model training.

In [29]:
# Lightweight load
text_embeddings = np.load(text_embeddings_path, mmap_mode="r")  # interaction-level
image_embeddings = torch.load(image_embeddings_aligned_path, map_location="cpu")  # item-level aligned
time_features = torch.load(time_features_path, map_location="cpu")  # interaction-level

num_text_rows, text_dim = text_embeddings.shape
num_image_rows, image_dim = image_embeddings.shape
num_time_rows, time_raw_dim = time_features.shape

In [30]:
print("- Precomputed Embeddings / Feature Shapes ")
print(f"Text embeddings:  shape = ({num_text_rows}, {text_dim})  [interaction-level]")
print(f"Image embeddings: shape = ({num_image_rows}, {image_dim}) [item-level aligned]")
print(f"Time features:    shape = ({num_time_rows}, {time_raw_dim}) [interaction-level]")

- Precomputed Embeddings / Feature Shapes 
Text embeddings:  shape = (3730329, 1536)  [interaction-level]
Image embeddings: shape = (59142, 768) [item-level aligned]
Time features:    shape = (3730329, 5) [interaction-level]


In [31]:
# Load time feature names
with open(time_feature_cols_path, "r") as f:
    time_feature_cols = json.load(f)

print("\nTime feature columns:", time_feature_cols)


Time feature columns: ['unix_time_norm', 'hour_sin', 'hour_cos', 'dayofweek', 'is_weekend']


In [32]:
# Load missing mask
image_missing_mask = torch.load(image_missing_mask_path, map_location="cpu")
print("Missing image embeddings (count):", image_missing_mask.sum().item())

Missing image embeddings (count): 2


In [33]:
# item-level alignment (image embeddings)
assert num_image_rows == item_vocab_size, (
    f"Mismatch: image_embeddings rows ({num_image_rows}) != item_vocab_size ({item_vocab_size})"
)

# interaction-level alignment (text embeddings + time features)
assert num_text_rows == num_time_rows, (
    f"Mismatch: text rows ({num_text_rows}) != time rows ({num_time_rows})"
)

# time feature column alignment
assert time_raw_dim == len(time_feature_cols), (
    f"Mismatch: time_raw_dim ({time_raw_dim}) != number of time feature columns ({len(time_feature_cols)})"
)

In [34]:
# Basic numerical checks
# Sample small slices to avoid scanning all 3.7M rows
sample_idx = np.random.randint(0, num_time_rows, size=2048)

text_sample = np.array(text_embeddings[sample_idx])
time_sample = time_features[sample_idx].numpy()
image_sample = image_embeddings[np.random.randint(0, num_image_rows, size=256)].numpy()

assert not np.isnan(text_sample).any(), "NaNs found in text embeddings sample."
assert not np.isnan(time_sample).any(), "NaNs found in time features sample."
assert not np.isnan(image_sample).any(), "NaNs found in image embeddings sample."

print("Sanity checks passed (item-level + interaction-level alignment confirmed).")

Sanity checks passed (item-level + interaction-level alignment confirmed).


### FeatureStore Implementation

This step implements a lightweight FeatureStore that centralizes access to all precomputed modality features:

*   Text embeddings are loaded via NumPy memmap to avoid RAM blow-up.
*   Image embeddings are loaded as a CPU tensor (aligned to full item vocabulary).
*   Time features are loaded as a CPU tensor (moved to GPU per batch during training).
*   ID triplets (user/item/scene indices) are loaded from the preprocessed df_id_encoded file.

The FeatureStore provides a consistent retrieval API used by CTRDataset, fusion modules, training, evaluation, and explainability.

In [37]:
# First we will Load df_id_encoded + validate required columns
# Prefer pickle (faster, preserves types), fallback to CSV
df_id_encoded_pkl = os.path.join(preprocessed_dir, "split_features", "df_id_encoded.pkl")
df_id_encoded_csv = os.path.join(preprocessed_dir, "split_features", "df_id_encoded.csv")

if os.path.exists(df_id_encoded_pkl):
    df_id = pd.read_pickle(df_id_encoded_pkl)
    print("Loaded df_id_encoded.pkl:", df_id.shape)
else:
    df_id = pd.read_csv(df_id_encoded_csv)
    print("Loaded df_id_encoded.csv:", df_id.shape)

# --- REQUIRED columns (must exist) ---
required_cols = ["user_id_idx", "item_id_idx", "scene_idx"]
missing_cols = [c for c in required_cols if c not in df_id.columns]

print("df_id columns:", list(df_id.columns))
assert len(missing_cols) == 0, f"Missing required columns in df_id_encoded: {missing_cols}"

# Optional label column (different pipelines may name it differently)
label_col_candidates = ["label", "is_click", "click", "y"]
label_col = next((c for c in label_col_candidates if c in df_id.columns), None)

print("Detected label column:", label_col)
df_id.head()

Loaded df_id_encoded.pkl: (3730329, 3)
df_id columns: ['user_id_idx', 'item_id_idx', 'scene_idx']
Detected label column: None


,user_id_idx,item_id_idx,scene_idx
0,41688,13590,4
1,39834,21351,2
2,22354,33398,2
3,17547,11190,1
4,55873,33398,2


In [42]:
class FeatureStore:
    """
    Centralized feature access layer for ColdFusionX.

    - text_embeddings: interaction-level (row_idx aligned)
    - time_features: interaction-level (row_idx aligned)
    - image_embeddings: item-level (item_idx aligned)
    - id mappings: optional (debug/traceability)
    """
    def __init__(
        self,
        text_embeddings_path: str,
        image_embeddings_path: str,
        time_features_path: str,
        df_id: pd.DataFrame,
        image_missing_mask_path: Optional[str] = None,
        id_maps: Optional[Dict[str, Dict]] = None
    ):
        # Interaction-level embeddings (memmap)
        self.text_embeddings = np.load(text_embeddings_path, mmap_mode="r")  # (N_rows, text_dim)

        # Item-level embeddings (aligned)
        self.image_embeddings = torch.load(image_embeddings_path, map_location="cpu")  # (num_items, 768)

        # Interaction-level time features
        self.time_features = torch.load(time_features_path, map_location="cpu")  # (N_rows, time_dim)

        # ID index dataframe (interaction-level)
        self.df_id = df_id.reset_index(drop=True)

        # Optional mask: True indicates missing image embedding
        self.image_missing_mask = None
        if image_missing_mask_path and os.path.exists(image_missing_mask_path):
            self.image_missing_mask = torch.load(image_missing_mask_path, map_location="cpu")

        # Optional ID mapping dicts (debugging)
        self.id_maps = id_maps or {}

        # Cache shapes for quick checks
        self.num_rows = len(self.df_id)
        self.text_dim = self.text_embeddings.shape[1]
        self.image_dim = self.image_embeddings.shape[1]
        self.time_dim = self.time_features.shape[1]

        # Strong alignment checks (interaction-level)
        assert self.text_embeddings.shape[0] == self.num_rows, (
            f"text_embeddings rows ({self.text_embeddings.shape[0]}) != df_id rows ({self.num_rows})"
        )
        assert self.time_features.shape[0] == self.num_rows, (
            f"time_features rows ({self.time_features.shape[0]}) != df_id rows ({self.num_rows})"
        )

    # --------------
    # API Methods
    # --------------
    def get_text(self, row_idx: int) -> np.ndarray:
        """Return interaction-level text vector for a given row index."""
        return np.array(self.text_embeddings[row_idx], dtype=np.float32)

    def get_time(self, row_idx: int) -> torch.Tensor:
        """Return interaction-level time features for a given row index."""
        return self.time_features[row_idx].float()

    def get_id_triplet(self, row_idx: int) -> Tuple[int, int, int]:
        """Return (user_id_idx, item_id_idx, scene_idx) for a given row index."""
        row = self.df_id.iloc[row_idx]
        return int(row["user_id_idx"]), int(row["item_id_idx"]), int(row["scene_idx"])

    def get_image(self, item_id_idx: int) -> torch.Tensor:
        """Return item-level image embedding for a given item index (aligned)."""
        return self.image_embeddings[item_id_idx].float()

    # -----------------------
    # Additional helpers
    # -----------------------
    def has_image(self, item_id_idx: int) -> bool:
        """True if image embedding was present (not missing)."""
        if self.image_missing_mask is None:
            return True
        return not bool(self.image_missing_mask[item_id_idx].item())

In [43]:
# Paths (using your aligned artifacts)
image_embeddings_aligned_path = os.path.join(base_dir, "image_embeddings_vit_base_aligned.pt")
image_missing_mask_path = os.path.join(base_dir, "image_missing_mask.pt")

# Optional: load id_maps for debugging (not required for runtime)
id_maps = {}
for name, path in [
    ("user_id_2idx", user_id_2idx_path),
    ("item_id_2idx", item_id_2idx_path),
    ("scene_2idx", scene_2idx_path),
]:
    with open(path, "r") as f:
        id_maps[name] = json.load(f)

feature_store = FeatureStore(
    text_embeddings_path=text_embeddings_path,
    image_embeddings_path=image_embeddings_aligned_path,
    time_features_path=time_features_path,
    df_id=df_id,
    image_missing_mask_path=image_missing_mask_path,
    id_maps=id_maps
)

print("FeatureStore created successfully.")
print("num_rows:", feature_store.num_rows)
print("text_dim:", feature_store.text_dim)
print("image_dim:", feature_store.image_dim)
print("time_dim:", feature_store.time_dim)

# --- Quick retrieval smoke test ---
test_row = 0
u_idx, i_idx, s_idx = feature_store.get_id_triplet(test_row)
text_vec = feature_store.get_text(test_row)
time_vec = feature_store.get_time(test_row)
img_vec = feature_store.get_image(i_idx)

print("\n--- Smoke Test ---")
print("row_idx:", test_row)
print("user_idx, item_idx, scene_idx:", (u_idx, i_idx, s_idx))
print("text_vec shape:", text_vec.shape)
print("time_vec shape:", tuple(time_vec.shape))
print("img_vec shape:", tuple(img_vec.shape))
print("has_image(item_idx):", feature_store.has_image(i_idx))

FeatureStore created successfully.
num_rows: 3730329
text_dim: 1536
image_dim: 768
time_dim: 5

--- Smoke Test ---
row_idx: 0
user_idx, item_idx, scene_idx: (41688, 13590, 4)
text_vec shape: (1536,)
time_vec shape: (5,)
img_vec shape: (768,)
has_image(item_idx): True


In [ ]:
!git status

Refresh index: 100% (14/14), done.
